# 🌳 Decision Trees
**ISLP Chapter 8 · Pattern Recognition for the Rest of Us**

> Decision trees split data by asking a sequence of yes/no questions. They're one of the most interpretable ML models — you can literally draw them out and explain them to anyone.

### What you'll learn
- How trees split using Gini impurity and RSS
- Growing and pruning trees to avoid overfitting
- Classification trees vs regression trees
- Tree visualization and interpretation
- Why trees are the building block for Random Forests and Boosting

### Dataset: Carseats (sales prediction) + Heart disease (classification)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Carseats

In [ ]:
import pandas as pd
carseats = pd.read_csv(f'{DATA_DIR}/Carseats.csv')
carseats = pd.get_dummies(carseats, drop_first=True, dtype=float)
X_reg = carseats.drop('Sales', axis=1)
y_reg = carseats['Sales']
print(f"Carseats: {X_reg.shape}  |  Predicting: Sales")

### Load Dataset: Heart

In [ ]:
import pandas as pd
heart = pd.read_csv(f'{DATA_DIR}/Heart.csv').dropna()
heart = pd.get_dummies(heart, drop_first=True, dtype=float)
target_col = [c for c in heart.columns if 'AHD' in c or c == heart.columns[-1]][0]
X = heart.drop(target_col, axis=1)
y = heart[target_col].astype(int)
print(f"Heart: {X.shape}  |  Disease rate: {y.mean():.1%}")

## 🌳 Part 1 — How Trees Split

**For regression trees:** At each node, find the feature j and threshold s that minimize RSS:
```
RSS = Σ(yᵢ - ȳ_left)² + Σ(yᵢ - ȳ_right)²
```

**For classification trees:** Minimize **Gini impurity** at each node:
```
Gini = 1 - Σₖ p̂ₖ²
```
where p̂ₖ is the proportion of class k observations in the node.
A node with all one class has Gini=0 (pure). A 50/50 split has Gini=0.5 (most impure).

In [ ]:
# Visualize Gini impurity
p = np.linspace(0.001, 0.999, 200)
gini = 1 - p**2 - (1-p)**2
entropy = -p*np.log2(p) - (1-p)*np.log2(1-p)
misclass = 1 - np.maximum(p, 1-p)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(p, gini,     color='#1e5fa8', lw=2, label='Gini impurity')
ax.plot(p, entropy/2, color='#e85d20', lw=2, ls='--', label='Entropy/2')
ax.plot(p, misclass, color='#888',    lw=1.5, ls=':', label='Misclassification rate')
ax.set_xlabel('Proportion of class 1 in node')
ax.set_ylabel('Impurity measure')
ax.set_title('Node Impurity Measures — all maximized at 50/50 split')
ax.legend()
ax.axvline(0.5, color='black', lw=0.8, ls='--', alpha=0.4)
plt.tight_layout()
plt.show()
print("📌 Trees try to create pure nodes (low Gini). A pure node = all observations are one class.")

In [ ]:
# Grow and visualize a classification tree
X_tr, X_te, y_tr, y_te = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

tree_clf = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
tree_clf.fit(X_tr, y_tr)

print(f"Training accuracy: {tree_clf.score(X_tr, y_tr):.3f}")
print(f"Test accuracy:     {tree_clf.score(X_te, y_te):.3f}")
print(f"Number of leaves:  {tree_clf.get_n_leaves()}")

fig, ax = plt.subplots(figsize=(20, 7))
plot_tree(tree_clf, feature_names=X_clf.columns, class_names=['No Disease','Disease'],
          filled=True, rounded=True, fontsize=8, ax=ax,
          impurity=True, proportion=False)
ax.set_title('Classification Tree (max_depth=4) — Heart Disease Prediction', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# The overfitting problem — training vs test accuracy by tree depth
depths = range(1, 20)
train_acc, test_acc = [], []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    t.fit(X_tr, y_tr)
    train_acc.append(t.score(X_tr, y_tr))
    test_acc.append(t.score(X_te, y_te))

best_depth = depths[test_acc.index(max(test_acc))]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(depths), train_acc, 'o-', color='#1e5fa8', lw=2, label='Training accuracy')
ax.plot(list(depths), test_acc,  's-', color='#e85d20', lw=2, label='Test accuracy')
ax.axvline(best_depth, color='#1a7a45', lw=1.5, ls='--', label=f'Best depth = {best_depth}')
ax.set_xlabel('Max Tree Depth')
ax.set_ylabel('Accuracy')
ax.set_title('Overfitting: Deep Trees Memorize Training Data')
ax.legend()
plt.tight_layout()
plt.show()
print(f"\n📌 Training accuracy → 100% as depth increases (memorizes every point)")
print(f"   Test accuracy peaks at depth {best_depth} then falls — overfitting begins")

In [ ]:
# Regression tree on Carseats
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)
tree_reg = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_reg.fit(X_tr_r, y_tr_r)

train_mse = mean_squared_error(y_tr_r, tree_reg.predict(X_tr_r))
test_mse  = mean_squared_error(y_te_r, tree_reg.predict(X_te_r))

print(f"Regression Tree (max_depth=4) — Carseats Sales")
print(f"Training RMSE: {np.sqrt(train_mse):.3f}")
print(f"Test RMSE:     {np.sqrt(test_mse):.3f}")

# Feature importance
imp = pd.Series(tree_reg.feature_importances_, index=X_reg.columns).sort_values(ascending=False).head(8)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(imp.index, imp.values, color='#1e5fa8', edgecolor='white')
ax.set_ylabel('Feature Importance')
ax.set_title('Regression Tree — Feature Importance for Carseats Sales')
ax.set_xticklabels(imp.index, rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
answers = {
    "q1": "",  # What is Gini impurity and what value indicates a pure node?
    "q2": "",  # Why do very deep trees overfit?
    "q3": "",  # What is the difference between pruning and setting max_depth?
    "q4": "",  # Name one advantage and one disadvantage of decision trees vs linear models
    "q5": "",  # Why are decision trees the building block for Random Forests and Boosting?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
# ── Submit your results to the instructor ────────────────────────────────────
# Fill in your GitHub username (do this once, it stays saved in the notebook)
GITHUB_USERNAME = ""   # ← put your GitHub username here e.g. "jsmith42"

# ─────────────────────────────────────────────────────────────────────────────
# DO NOT EDIT BELOW THIS LINE
import json, urllib.parse

def build_submission_url(username, notebook_name, score, total, answers_dict,
                          form_base_url=None, entries=None):
    """Build a pre-filled Google Form URL for quiz submission."""
    if not form_base_url:
        print("⚠  No submission URL configured yet.")
        print("   Your instructor will share the Form URL — paste it into form_base_url below.")
        print("   Your answers are saved in this notebook regardless.")
        return None

    params = {
        entries.get('user',     'entry.000000001'): username,
        entries.get('notebook', 'entry.000000002'): notebook_name,
        entries.get('score',    'entry.000000003'): str(score),
        entries.get('total',    'entry.000000004'): str(total),
    }
    url = form_base_url.replace('/viewform','') + '/viewform?' + urllib.parse.urlencode(params)
    return url

# ── Submission config (your instructor fills these in) ───────────────────────
FORM_BASE_URL = None   # e.g. "https://docs.google.com/forms/d/e/XXXX/viewform"
FORM_ENTRIES  = {      # entry IDs from the Google Form
    'user':     'entry.000000001',
    'notebook': 'entry.000000002',
    'score':    'entry.000000003',
    'total':    'entry.000000004',
}
# ─────────────────────────────────────────────────────────────────────────────

# Calculate score from answers dict
score_val = sum(1 for v in answers.values() if str(v).strip())  # counts answered Qs
# For auto-graded notebooks the score is passed directly
# Here we just verify completion
n_answered = sum(1 for v in answers.values() if str(v).strip())
n_total    = len(answers)

print(f"\n{'='*50}")
print(f"Quiz completion: {n_answered}/{n_total} questions answered")
if n_answered < n_total:
    print(f"⚠  Please answer all {n_total} questions before submitting.")
else:
    print("✅ All questions answered!")
    if GITHUB_USERNAME:
        import os
        nb_name = os.path.basename(globals().get('__vsc_ipynb_file__',
                  globals().get('__file__', 'unknown-notebook'))).replace('.ipynb','')
        url = build_submission_url(GITHUB_USERNAME, nb_name,
                                   n_answered, n_total, answers,
                                   FORM_BASE_URL, FORM_ENTRIES)
        if url:
            print(f"\n🔗 Submit to instructor:")
            print(f"   {url}")
            print("\n   Copy the URL above, open it in a new tab, and click Submit.")
        else:
            print("\n   Save your notebook to GitHub to record your progress:")
            print("   File → Save a copy in GitHub → choose your fork")
    else:
        print("\n⚠  Add your GitHub username to GITHUB_USERNAME above, then re-run.")
    print(f"{'='*50}")